### **index_document.ipynb**
### **Index document pipeline**

* ##### 01 - Install packages
* ##### 02 - Import packages
* ##### 03 - Create tasks
* ##### 04 - Create pipeline
* ##### 05 - Create pipeline yaml
* ##### 06 - Create pipeline run

### 01 - Install packages

In [ ]:
from sys import executable

In [ ]:
!{ executable } -m pip install --upgrade kfp[kubernetes]

### 02 - Import packages

In [ ]:
from kfp          import kubernetes
from kfp.client   import Client
from kfp.compiler import Compiler
from kfp.dsl      import component, pipeline
from os           import path

### 03 - Create tasks

In [ ]:
task_base_image = 'registry.access.redhat.com/ubi9/python-311'

In [ ]:
@component(
    base_image          = task_base_image,
    packages_to_install = ['boto3']
)
def download_document(
    s3_service_name      : str,
    s3_endpoint_url      : str,
    s3_access_key_id     : str,
    s3_secret_access_key : str,
    s3_region            : str,
    s3_bucket            : str,
    s3_file              : str,
    pvc_directory        : str
) -> str:

    from boto3 import client
    from os    import makedirs, path

    pvc_directory = path.join(pvc_directory, 'documents')
    pvc_file      = path.join(pvc_directory, path.basename(s3_file))

    makedirs(
        name     = pvc_directory,
        exist_ok = True
    )

    s3_client = client(
        service_name          = s3_service_name,
        endpoint_url          = s3_endpoint_url,
        aws_access_key_id     = s3_access_key_id,
        aws_secret_access_key = s3_secret_access_key,
        region_name           = s3_region
    )

    s3_client.download_file(s3_bucket, s3_file, pvc_file)

    return pvc_file

In [ ]:
@component(
    base_image          = task_base_image,
    packages_to_install = [
        'elasticsearch',
        'langchain',
        'langchain-community',
        'langchain-elasticsearch',
        'langchain-openai',
        'pypdf'
    ]
)
def index_document(
    elasticsearch_host     : str,
    elasticsearch_username : str,
    elasticsearch_password : str,
    elasticsearch_index    : str,
    pvc_file               : str
):

    from elasticsearch                        import Elasticsearch
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_elasticsearch              import ElasticsearchStore
    from langchain_openai                     import OpenAIEmbeddings

    loader    = PyPDFLoader(pvc_file)
    documents = loader.load_and_split()

    elasticsearch_client = Elasticsearch(
        host         = elasticsearch_host,
        basic_auth   = (elasticsearch_username, elasticsearch_password),
        verify_certs = False
    )

    embedding = OpenAIEmbeddings()

    elasticsearch_store = ElasticsearchStore(
        es_connection = elasticsearch_client,
        index_name    = elasticsearch_index,
        embedding     = embedding
    )

    elasticsearch_store.from_documents(
        documents = documents
    )

### 04 - Create pipeline

In [ ]:
pipeline_name        = 'index_document'
pipeline_description = 'Index document pipeline'

In [ ]:
@pipeline(
    name        = pipeline_name,
    description = pipeline_description
)
def index_document_pipeline(
    pvc_name               : str,
    pvc_storage_class      : str,
    pvc_access_modes       : list,
    pvc_size               : str,
    s3_service_name        : str,
    s3_endpoint_url        : str,
    s3_access_key_id       : str,
    s3_secret_access_key   : str,
    s3_region              : str,
    s3_bucket              : str,
    s3_file                : str,
    elasticsearch_host     : str,
    elasticsearch_username : str,
    elasticsearch_password : str,
    elasticsearch_index    : str
):

    # TASK Create PVC

    pvc_directory = path.join('/', pipeline_name)

    create_pvc_task = kubernetes.CreatePVC(
        pvc_name           = pvc_name,
        storage_class_name = pvc_storage_class,
        access_modes       = pvc_access_modes,
        size               = pvc_size
    )

    # TASK Download document

    download_document_task = download_document(
        s3_service_name      = s3_service_name,
        s3_endpoint_url      = s3_endpoint_url,
        s3_access_key_id     = s3_access_key_id,
        s3_secret_access_key = s3_secret_access_key,
        s3_region            = s3_region,
        s3_bucket            = s3_bucket,
        s3_file              = s3_file,
        pvc_directory        = pvc_directory
    )

    download_document_task.after(create_pvc_task)

    kubernetes.mount_pvc(
        task       = download_document_task,
        pvc_name   = pvc_name,
        mount_path = pvc_directory
    )

    pvc_file = download_document_task.output

    # TASK Index document

    index_document_task = index_document(
        elasticsearch_host     = elasticsearch_host,
        elasticsearch_username = elasticsearch_username,
        elasticsearch_password = elasticsearch_password,
        elasticsearch_index    = elasticsearch_index,
        pvc_file               = pvc_file
    )

    index_document_task.after(download_document_task)

    kubernetes.mount_pvc(
        task       = index_document_task,
        pvc_name   = pvc_name,
        mount_path = pvc_directory
    )

### 05 - Create pipeline yaml

In [ ]:
pipeline_yaml = path.join(f'{ pipeline_name }.yaml')

In [ ]:
Compiler().compile(
    pipeline_func = index_document_pipeline,
    package_path  = pipeline_yaml
)

### 06 - Create pipeline run

In [ ]:
kubeflow_host = '<kubeflow_host>'

In [ ]:
pipeline_arguments = {
    'pvc_name'               : '<pvc_name>',
    'pvc_storage_class'      : '<pvc_storage_class>',
    'pvc_access_modes'       : ['<pvc_access_modes>'],
    'pvc_size'               : '<pvc_size>',
    's3_service_name'        : '<s3_service_name>',
    's3_endpoint_url'        : '<s3_endpoint_url>',
    's3_access_key_id'       : '<s3_access_key_id>',
    's3_secret_access_key'   : '<s3_secret_access_key>',
    's3_region'              : '<s3_region>',
    's3_bucket'              : '<s3_bucket>',
    's3_file'                : '<s3_file>',
    'elasticsearch_host'     : '<elasticsearch_host>',
    'elasticsearch_username' : '<elasticsearch_username>',
    'elasticsearch_password' : '<elasticsearch_password>',
    'elasticsearch_index'    : '<elasticsearch_index>'
}

In [ ]:
Client(host = kubeflow_host).create_run_from_pipeline_package(
    pipeline_file = pipeline_yaml,
    arguments     = pipeline_arguments
)